# 基金选股择时能力定量评价模型
## T-M 模型 / H-M 模型 / C-L 模型

**华泰金工研究 | 2020-08-21**

---

本notebook复现华泰金工研报《定量评价基金的选股择时能力》中的三种经典模型：

1. **T-M模型** (Treynor-Mazuy, 1966)：在詹森alpha指数模型中引入二次项
2. **H-M模型** (Henriksson-Merton, 1981)：引入虚拟变量区分牛熊市
3. **C-L模型** (Chang-Lewellen, 1984)：区分多头与空头市场

---

In [ ]:
# -*- coding: utf-8 -*-
"""环境初始化"""
import sys
import os

# 添加项目根目录到路径
project_root = os.path.dirname(os.path.abspath('.'))
sys.path.insert(0, project_root)

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

# 字体设置（先style后字体）
plt.style.use('seaborn-v0_8-whitegrid')
matplotlib.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False
matplotlib.rcParams['figure.dpi'] = 120

print('环境初始化完成')

## 1. 导入项目模块

In [ ]:
from source.data_loader import load_all_data, load_fund_nav, load_benchmark
from source.factor import TMModel, HMModel, CLModel, StockTimingEvaluator
from source.backtest import RollingTimingBacktest, PerformanceAttribution
from source.plot import (
    plot_timing_dashboard, plot_rolling_timing,
    plot_excess_return_scatter, plot_attribution_area
)
from source.utils import summary_stats, get_fund_name
from config import START_DATE, END_DATE, BENCHMARK, RISK_FREE_RATE

print('模块导入完成')

## 2. 配置参数

In [ ]:
# ========== 可修改参数 ==========
FUND_CODE   = '021181'       # 基金代码
START_DATE  = '2021-01-01'   # 开始日期
END_DATE    = '2026-04-28'   # 结束日期
BENCHMARK   = '000300.SH'    # 基准：沪深300
RISK_FREE_RATE = 0.015      # 无风险利率（年化1.5%）

ROLLING_WINDOW = 252          # 滚动窗口（1年，交易日）
ROLLING_STEP   = 21          # 滚动步长（月度，21个交易日）

OUTPUT_DIR = os.path.join(os.path.dirname(os.path.abspath('.')), 'output', FUND_CODE)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'基金代码: {FUND_CODE} ({get_fund_name(FUND_CODE)})')
print(f'分析区间: {START_DATE} 至 {END_DATE}')
print(f'基准指数: {BENCHMARK}')
print(f'输出目录: {OUTPUT_DIR}')

## 3. 数据加载

In [ ]:
print('正在加载数据（efinance优先）...')
fund_returns, bench_returns = load_all_data(
    fund_code=FUND_CODE,
    start_date=START_DATE,
    end_date=END_DATE,
    benchmark=BENCHMARK
)

print(f'\n数据加载完成：{len(fund_returns)} 个交易日')
print(fund_returns.head())

## 4. 基础统计

In [ ]:
fund_stats = summary_stats(fund_returns['基金收益率'])
bench_stats = summary_stats(bench_returns['基准收益率'])

print('【基金业绩统计】')
print(f'  年化收益率: {fund_stats["ann_return"]*100:.2f}%')
print(f'  年化波动率: {fund_stats["ann_vol"]*100:.2f}%')
print(f'  夏普比率:   {fund_stats["sharpe"]:.3f}')
print(f'  最大回撤:   {fund_stats["max_drawdown"]*100:.2f}%')

print('\n【基准指数统计】')
print(f'  年化收益率: {bench_stats["ann_return"]*100:.2f}%')
print(f'  年化波动率: {bench_stats["ann_vol"]*100:.2f}%')
print(f'  夏普比率:   {bench_stats["sharpe"]:.3f}')

## 5. 三种模型回归分析

In [ ]:
evaluator = StockTimingEvaluator(
    fund_returns['基金收益率'],
    bench_returns['基准收益率'],
    risk_free_rate=RISK_FREE_RATE
)
model_results = evaluator.evaluate()

# 汇总表
summary_df = evaluator.get_summary()
print('\n模型结果汇总表:')
display(summary_df)

## 6. 各模型详细结果

In [ ]:
# T-M 模型
print('='*60)
print('T-M 模型 (Treynor-Mazuy, 1966)')
print('公式: Rp - Rf = alpha + beta1*(Rm-Rf) + beta2*(Rm-Rf)^2 + epsilon')
print('='*60)
tm_res = model_results.get('TM', {})
if tm_res:
    print(f"Alpha (选股能力):   {tm_res['alpha']:.6f}  (p={tm_res['alpha_pvalue']:.4f})")
    print(f"Beta1 (市场风险):   {tm_res['beta1']:.6f}")
    print(f"Beta2 (择时能力):   {tm_res['beta2']:.6f}  (p={tm_res['beta2_pvalue']:.4f})")
    print(f"择时能力判断: {'有' if tm_res['timing_ability'] else '无'} (beta2>0 且 p<0.05)")
    print(f"R-squared: {tm_res['r_squared']:.4f}")

In [ ]:
# H-M 模型
print('='*60)
print('H-M 模型 (Henriksson-Merton, 1981)')
print('公式: Rp - Rf = alpha + beta1*(Rm-Rf) + beta2*(Rm-Rf)*D + epsilon')
print('       D=1牛市( Rm>Rf ), D=0熊市( Rm<Rf )')
print('='*60)
hm_res = model_results.get('HM', {})
if hm_res:
    print(f"Alpha (选股能力):   {hm_res['alpha']:.6f}  (p={hm_res['alpha_pvalue']:.4f})")
    print(f"Beta1 (熊市Beta):   {hm_res['beta1']:.6f}")
    print(f"Beta2 (择时增量):   {hm_res['beta2']:.6f}  (p={hm_res['beta2_pvalue']:.4f})")
    print(f"牛市Beta(beta1+beta2): {hm_res['bull_beta']:.6f}")
    print(f"熊市Beta(beta1):     {hm_res['bear_beta']:.6f}")
    print(f"择时能力判断: {'有' if hm_res['timing_ability'] else '无'} (beta2>0 且 p<0.05)")
    print(f"R-squared: {hm_res['r_squared']:.4f}")

In [ ]:
# C-L 模型
print('='*60)
print('C-L 模型 (Chang-Lewellen, 1984)')
print('公式: Rp-Rf = alpha + beta1*(Rm-Rf)*D1 + beta2*(Rm-Rf)*D2 + epsilon')
print('       多头(Rm>Rf): D1=0, D2=1; 空头(Rm<Rf): D1=1, D2=0')
print('='*60)
cl_res = model_results.get('CL', {})
if cl_res:
    print(f"Alpha (选股能力):   {cl_res['alpha']:.6f}  (p={cl_res['alpha_pvalue']:.4f})")
    print(f"Beta1 (空头Beta):   {cl_res['beta1']:.6f}")
    print(f"Beta2 (多头Beta):   {cl_res['beta2']:.6f}")
    print(f"差值(beta2-beta1):  {cl_res['timing_diff']:.6f}")
    print(f"择时能力判断: {'有' if cl_res['timing_ability'] else '无'} (beta2-beta1>0)")
    print(f"R-squared: {cl_res['r_squared']:.4f}")

## 7. 滚动回测（时序分析）

In [ ]:
print(f'运行滚动回测: window={ROLLING_WINDOW}天, step={ROLLING_STEP}天...')

bt = RollingTimingBacktest(
    fund_returns['基金收益率'],
    bench_returns['基准收益率'],
    window=ROLLING_WINDOW,
    risk_free_rate=RISK_FREE_RATE
)

rolling_tm = bt.run('TM', step=ROLLING_STEP)
rolling_hm = bt.run('HM', step=ROLLING_STEP)
rolling_cl = bt.run('CL', step=ROLLING_STEP)

print(f'T-M 滚动窗口数: {len(rolling_tm)}')
print(f'H-M 滚动窗口数: {len(rolling_hm)}')
print(f'C-L 滚动窗口数: {len(rolling_cl)}')

In [ ]:
# T-M 滚动结果预览
print('T-M 滚动回归结果（最近10期）:')
display(rolling_tm.tail(10))

## 8. 可视化图表

In [ ]:
# 仪表盘图
dashboard_data = dict(model_results)
dashboard_data['excess_fund'] = fund_returns['基金收益率']
dashboard_data['excess_bench'] = bench_returns['基准收益率']

dashboard_path = os.path.join(OUTPUT_DIR, f'{FUND_CODE}_timing_dashboard.png')
plot_timing_dashboard(
    fund_code=FUND_CODE,
    fund_name=get_fund_name(FUND_CODE),
    model_results=dashboard_data,
    output_path=dashboard_path
)
print(f'仪表盘已保存: {dashboard_path}')

In [ ]:
# T-M 滚动时序图
if not rolling_tm.empty:
    tm_rolling_path = os.path.join(OUTPUT_DIR, f'{FUND_CODE}_rolling_TM.png')
    plot_rolling_timing(
        rolling_tm, 'TM', tm_rolling_path,
        title=f'{FUND_CODE} {get_fund_name(FUND_CODE)} - T-M Model Rolling Timing'
    )
    print(f'T-M 滚动图已保存: {tm_rolling_path}')

## 9. 保存结果

In [ ]:
import json

# 保存JSON结果
output_data = {
    'fund_code': FUND_CODE,
    'fund_name': get_fund_name(FUND_CODE),
    'analysis_period': {'start': START_DATE, 'end': END_DATE},
    'benchmark': BENCHMARK,
    'risk_free_rate': RISK_FREE_RATE,
    'data_points': int(len(fund_returns)),
    'fund_stats': {k: float(v) for k, v in fund_stats.items()},
    'model_results': {
        k: {kk: float(vv) if isinstance(vv, (np.floating, np.integer)) else vv
            for kk, vv in v.items()}
        for k, v in model_results.items()
    }
}

json_path = os.path.join(OUTPUT_DIR, f'{FUND_CODE}_timing_results.json')
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(output_data, f, ensure_ascii=False, indent=2)

print(f'JSON结果已保存: {json_path}')

# 保存滚动CSV
for name, df in [('TM', rolling_tm), ('HM', rolling_hm), ('CL', rolling_cl)]:
    if not df.empty:
        csv_path = os.path.join(OUTPUT_DIR, f'{FUND_CODE}_rolling_{name}.csv')
        df.to_csv(csv_path)
        print(f'滚动CSV({name})已保存: {csv_path}')

## 10. 结论与解读

In [ ]:
# 综合判断
timing_count = sum(1 for r in model_results.values() if r.get('timing_ability', False))
stock_count = sum(1 for r in model_results.values() if r.get('stock_ability', False))

print('='*60)
print(f'基金: {get_fund_name(FUND_CODE)} ({FUND_CODE})')
print('='*60)
print(f'\n【综合判断】')

if timing_count >= 2:
    print(f'  择时能力: 确认（{timing_count}/3 模型显著）')
elif timing_count == 1:
    print(f'  择时能力: 弱确认（{timing_count}/3 模型显著）')
else:
    print(f'  择时能力: 未确认（0/3 模型显著）')

if stock_count >= 2:
    print(f'  选股能力: 确认（{stock_count}/3 模型显著）')
elif stock_count == 1:
    print(f'  选股能力: 弱确认（{stock_count}/3 模型显著）')
else:
    print(f'  选股能力: 未确认（0/3 模型显著）')

print(f'\n分析区间: {START_DATE} 至 {END_DATE} ({len(fund_returns)}个交易日)')
print(f'基金年化收益: {fund_stats["ann_return"]*100:.2f}%')
print(f'基金夏普比率: {fund_stats["sharpe"]:.3f}')
print('='*60)